In [1]:
import sys, os

module_dir = '/home/ubuntu/mvm-accelerator/sw/model' 

if module_dir not in sys.path:
    sys.path.append(module_dir)

import numpy as np
import scipy.io as sio
from pathlib import Path    

from init import initialize
from results import results
from backward import backward
from plots import plots

In [2]:
OFFLOAD = True # True <=> use RTL logic for MVM

# Initialize model
state = initialize()

if OFFLOAD:
    from kernel import MVMKernel
    state.Npad = 17088
    state.kernel = MVMKernel(
        bitfile=os.path.join(module_dir, "../../hw/fp32_17088x17088_250/design_1.bit"),
        matrix_path=os.path.join(module_dir, "Data/trmult_reduced32_padded_shaped.npy"),
        Npad=state.Npad,
        element_width_bits=32,
        file_type="npy",
        verbose=1
)

# Distribution of land for simulation
H0_arr = np.asarray(state.H0).reshape(-1)
H = H0_arr[state.earth_indices]

# Number of periods
nb_per = 600

# Number of periods for backward simulation
nb_back = 180

[kernel] dtype: <class 'numpy.float32'>
[kernel] Npad: 17088
[kernel] channels: 4
[kernel] rows_per_channel: 4272
[kernel] elements_per_row: 17088
[kernel] elements_per_partition: 4272
[kernel] bytes_per_partition: 17088
[kernel] partition_base: ['0x80000000', '0x80008000', '0x80010000', '0x80018000']
[kernel] matrix CMA allocation FAILED: tile_rows=4272 -> halving
[kernel] matrix CMA allocation OK: tile_rows=2136


In [3]:
# Run the model and obtain summary statistics
results_data = results(H, nb_per, state)
realgdp_w, u_w, u2_w, prod_w, phi_w, PDV_u_w, PDV_u2_w, PDV_realgdp_w, migr_cell, migr_ctry, l, u, u2, tau, realgdp = results_data

t=1
[[1.5906524e-18 5.0071861e-18 7.2242886e-18 ... 6.2171663e-14
  3.9601148e-12 5.1095825e-12]
 [1.9216365e-12 4.5116467e-12 8.7033306e-12 ... 1.1736150e-19
  1.1838962e-21 3.5630434e-22]
 [4.8440622e-21 8.8768532e-21 7.2270390e-21 ... 2.0910965e-24
  4.2445603e-25 1.2216336e-25]
 [5.1693636e-26 6.8379308e-26 7.0821066e-26 ... 0.0000000e+00
  0.0000000e+00 0.0000000e+00]]
[[[7.7024885e-03 9.1004808e-04 4.1691306e-05 ... 0.0000000e+00
   0.0000000e+00 0.0000000e+00]
  [9.1004808e-04 5.8322269e-02 1.4885439e-04 ... 0.0000000e+00
   0.0000000e+00 0.0000000e+00]
  [4.1691306e-05 1.4885439e-04 1.3198406e-03 ... 0.0000000e+00
   0.0000000e+00 0.0000000e+00]
  ...
  [9.8130833e-08 9.6644108e-08 8.4852928e-08 ... 0.0000000e+00
   0.0000000e+00 0.0000000e+00]
  [9.3641262e-08 9.2219352e-08 8.1209123e-08 ... 0.0000000e+00
   0.0000000e+00 0.0000000e+00]
  [8.6691394e-08 8.5144869e-08 7.5183969e-08 ... 0.0000000e+00
   0.0000000e+00 0.0000000e+00]]

 [[9.0976307e-08 8.9002782e-08 7.8381284e-08 

SystemExit: 1

/usr/local/share/pynq-venv/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3406: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
# Plot time series and maps, and save them
plots(H, realgdp_w, u_w, u2_w, prod_w, l, u, tau, realgdp)

In [ ]:
# Run model backwards
l_b, u_b, w_b, tau_b, phi_b, realgdp_b = backward(H, nb_back, state)

In [ ]:
# Calculate correlations
def calculate_correlation(x, y):
    return np.corrcoef(x, y)[0, 1]

pop0 = state.pop0
popminus5 = state.popminus5
popminus10 = state.popminus10
earth_indices = state.earth_indices

print('CORRELATIONS WITH 1995 DATA - CELL LEVEL')
print(calculate_correlation(popminus5[earth_indices], H0_arr[earth_indices] * l_b[:, 4]))
print(calculate_correlation(np.log(popminus5[earth_indices]), np.log(H0_arr[earth_indices] * l_b[:, 4])))
print(calculate_correlation(pop0[earth_indices] - popminus5[earth_indices], pop0[earth_indices] - H0_arr[earth_indices] * l_b[:, 4]))
print(calculate_correlation(np.log(pop0[earth_indices]) - np.log(popminus5[earth_indices]), np.log(pop0[earth_indices]) - np.log(H0_arr[earth_indices] * l_b[:, 4])))

print('CORRELATIONS WITH 1990 DATA - CELL LEVEL')
print(calculate_correlation(popminus10[earth_indices], H0_arr[earth_indices] * l_b[:, 9]))
print(calculate_correlation(np.log(popminus10[earth_indices]), np.log(H0_arr[earth_indices] * l_b[:, 9])))
print(calculate_correlation(pop0[earth_indices] - popminus10[earth_indices], pop0[earth_indices] - H0_arr[earth_indices] * l_b[:, 9]))
print(calculate_correlation(np.log(pop0[earth_indices]) - np.log(popminus10[earth_indices]), np.log(pop0[earth_indices]) - np.log(H0_arr[earth_indices] * l_b[:, 9])))

print('CORRELATIONS WITH 1995 DATA - COUNTRY LEVEL')
ctry_idx = state.C_vect.astype(int) - 1
popminus5_ctry_d = np.bincount(ctry_idx, weights=popminus5[earth_indices])
popminus5_ctry_m = np.bincount(ctry_idx, weights=H0_arr[earth_indices] * l_b[:, 4])
pop0_ctry = np.bincount(ctry_idx, weights=pop0[earth_indices])
print(calculate_correlation(popminus5_ctry_d, popminus5_ctry_m))
print(calculate_correlation(np.log(popminus5_ctry_d), np.log(popminus5_ctry_m)))
print(calculate_correlation(pop0_ctry - popminus5_ctry_d, pop0_ctry - popminus5_ctry_m))
print(calculate_correlation(np.log(pop0_ctry) - np.log(popminus5_ctry_d), np.log(pop0_ctry) - np.log(popminus5_ctry_m)))

print('CORRELATIONS WITH 1990 DATA - COUNTRY LEVEL')
popminus10_ctry_d = np.bincount(ctry_idx, weights=popminus10[earth_indices])
popminus10_ctry_m = np.bincount(ctry_idx, weights=H0_arr[earth_indices] * l_b[:, 9])
print(calculate_correlation(popminus10_ctry_d, popminus10_ctry_m))
print(calculate_correlation(np.log(popminus10_ctry_d), np.log(popminus10_ctry_m)))
print(calculate_correlation(pop0_ctry - popminus10_ctry_d, pop0_ctry - popminus10_ctry_m))
print(calculate_correlation(np.log(pop0_ctry) - np.log(popminus10_ctry_d), np.log(pop0_ctry) - np.log(popminus10_ctry_m)))

# Save all the output to disk
Path(os.path.join(module_dir, 'Output')).mkdir(parents=True, exist_ok=True)
sio.savemat(os.path.join(module_dir, 'Output/realgdp_w.mat'), {'realgdp_w': realgdp_w})
sio.savemat(os.path.join(module_dir, 'Output/u_w.mat'), {'u_w': u_w})
sio.savemat(os.path.join(module_dir, 'Output/u2_w.mat'), {'u2_w': u2_w})
sio.savemat(os.path.join(module_dir, 'Output/prod_w.mat'), {'prod_w': prod_w})
sio.savemat(os.path.join(module_dir, 'Output/phi_w.mat'), {'phi_w': phi_w})
sio.savemat(os.path.join(module_dir, 'Output/PDV_u_w.mat'), {'PDV_u_w': PDV_u_w})
sio.savemat(os.path.join(module_dir, 'Output/PDV_u2_w.mat'), {'PDV_u2_w': PDV_u2_w})
sio.savemat(os.path.join(module_dir, 'Output/PDV_realgdp_w.mat'), {'PDV_realgdp_w': PDV_realgdp_w})
sio.savemat(os.path.join(module_dir, 'Output/migr_cell.mat'), {'migr_cell': migr_cell})
sio.savemat(os.path.join(module_dir, 'Output/migr_ctry.mat'), {'migr_ctry': migr_ctry})
sio.savemat(os.path.join(module_dir, 'Output/l.mat'), {'l': l})
sio.savemat(os.path.join(module_dir, 'Output/u.mat'), {'u': u})
sio.savemat(os.path.join(module_dir, 'Output/realgdp.mat'), {'realgdp': realgdp})
sio.savemat(os.path.join(module_dir, 'Output/tau.mat'), {'tau': tau})
sio.savemat(os.path.join(module_dir, 'Output/l_b.mat'), {'l_b': l_b})

In [4]:
# Free CMA buffers
state.kernel.close()